In [1]:
import os

import matplotlib.pyplot as plt

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import numpy.typing as npt
import jax
import jax.numpy as jnp
from typing import List, Callable, Iterable, Tuple

from gridops_multidim import BSplineInterpolationAxis, BSplineInterpolationGrid
from gridops_multidim import set_up_grid_axis

from gridops_multidim import make_multi_indices_one_particle, arbitrary_dim_outer
from gridops_multidim import create_anterpolation_operator
from gridops_multidim import create_restriction_operator, create_prolongation_operator, \
    create_interaction_operator
from gridops_multidim import create_compute_U_oneplus, create_compute_U_and_f_oneplus

import sys

sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmfornn.splines.nesting import compute_J_zeroplus


In [2]:
%matplotlib notebook

# Basic settings

In [3]:
# geometry
length = 10.0
ndim = 3

# MSM
max_gridlevel = 4

# splines
p = 6
order = p - 1

# particles
n_particles = 100

In [4]:
J_zeroplus = compute_J_zeroplus(p)
J = jnp.concatenate((J_zeroplus[::-1][:-1], J_zeroplus))

# Create particle configuration

In [5]:
# TODO: change back to more particles and random charge
#  (few particles and hard-coded charges serve visualization purposes only)

rng = onp.random.default_rng(1632794)
pos = rng.uniform(0., length, size=(n_particles, ndim))
chg = rng.uniform(-1., 1., size=n_particles)
# chg = onp.array([-2, -2, 1, 1, 1])

# Construct grids

In [6]:
grid_axes_all_levels = [None]  # there is no grid at level zero
for l in range(1, max_gridlevel + 1):
    h = length / (2 ** (max_gridlevel - l))
    print(l, h)
    grid_axis = set_up_grid_axis(length=length, h=h, p=p, J_zeroplus=J_zeroplus, periodic=False)
    grid_axes_all_levels.append(grid_axis)
    
# For now, we are just replicating the same axis along all dimensions
# (i.e., same grid spacing, box size, and boundary conditions along all dimensions)
grids_all_levels = [(None, ) * ndim] + [BSplineInterpolationGrid((ga,) * ndim) for ga in grid_axes_all_levels[1:]]
grid_level_one = grids_all_levels[1]
grid_level_two = grids_all_levels[2]

1 1.25
2 2.5
3 5.0
4 10.0


# Functions

## Anterpolation

In [7]:
grid_level_one.shape

(15, 15, 15)

In [8]:
@jax.jit
def evaluate_bspline_basis_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_multiparticle(pos)


@jax.jit
def evaluate_bspline_basis_gradient_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_gradient_multiparticle(pos)

In [9]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_multiparticle(pos)

206 µs ± 41.4 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_gradient_multiparticle(pos)

413 µs ± 39 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
vals, inds = evaluate_bspline_basis_multiparticle(pos)
grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)

In [12]:
@jax.jit
def combined(pos):
    vals, inds = evaluate_bspline_basis_multiparticle(pos)
    grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)
    return inds, vals, grads

In [13]:
jax.device_put(pos)
%timeit combined(pos)

496 µs ± 26.3 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
anterpolate_level_one = create_anterpolation_operator(grid=grid_level_one)

In [15]:
jitted_anterpolate = jax.jit(anterpolate_level_one)

In [16]:
jax.device_put(pos)
jax.device_put(chg)

%timeit jitted_anterpolate(pos, chg).block_until_ready()

639 µs ± 77.4 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
gridcharge_level_one = anterpolate_level_one(pos, chg)

In [18]:
gridcharge_level_one

Array([0., 0., 0., ..., 0., 0., 0.], dtype=float64)

In [19]:
flat_indices = jnp.arange(gridcharge_level_one.shape[0])
unraveled_indices = jnp.unravel_index(flat_indices, grid_level_one.shape)
grid_points = []
for i, axis in enumerate(grid_level_one.axes):
    points = axis.to_raw_indices(unraveled_indices[i]) * axis.h
    grid_points.append(points)

In [20]:
mask = onp.abs(gridcharge_level_one) >= 0.01

x, y, z, = grid_points

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.scatter(x[mask], y[mask], z[mask], c=gridcharge_level_one[mask], s=10)
ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=chg, s=100)

plt.show()

<IPython.core.display.Javascript object>

## Restriction

In [21]:
def get_neigbhor_inds_on_fine_axis(
    idx_target: int,
    axis_source_fine: BSplineInterpolationAxis,
    axis_target_coarse: BSplineInterpolationAxis,
) -> jax.Array:
    """Get a target-grid index's neighbor indices on source grid."""
    raw_idx_targetgrid = axis_target_coarse.to_raw_indices(idx_target)
    raw_neighbor_inds_sourcegrid = 2 * raw_idx_targetgrid + jnp.arange(
        -p // 2, p // 2 + 1
    )
    neighbor_inds_sourcegrid = axis_source_fine.from_raw_indices(
        raw_neighbor_inds_sourcegrid
    )
    return neighbor_inds_sourcegrid

In [22]:
axis_level_1 = grid_axes_all_levels[1]
axis_level_2 = grid_axes_all_levels[2]

In [23]:
jnp.arange(axis_level_1.n_total)

Array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14], dtype=int64)

In [24]:
jnp.arange(axis_level_2.n_total)

Array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10], dtype=int64)

In [25]:
get_neigbhor_inds_on_fine_axis(0, axis_source_fine=axis_level_1, axis_target_coarse=axis_level_2)

Array([-6, -5, -4, -3, -2, -1,  0], dtype=int64)

In [26]:
from functools import partial

In [27]:
get_nb_inds_on_fa = partial(get_neigbhor_inds_on_fine_axis, axis_source_fine=grids_all_levels[1].axes[0], axis_target_coarse=grids_all_levels[2].axes[0])

In [28]:
get_nb_inds_on_fa = partial(get_neigbhor_inds_on_fine_axis, axis_source_fine=grids_all_levels[1].axes[1], axis_target_coarse=grids_all_levels[2].axes[1])

In [29]:
jax.vmap(get_nb_inds_on_fa)(jnp.arange(grids_all_levels[2].shape[0]))

Array([[-6, -5, -4, -3, -2, -1,  0],
       [-4, -3, -2, -1,  0,  1,  2],
       [-2, -1,  0,  1,  2,  3,  4],
       [ 0,  1,  2,  3,  4,  5,  6],
       [ 2,  3,  4,  5,  6,  7,  8],
       [ 4,  5,  6,  7,  8,  9, 10],
       [ 6,  7,  8,  9, 10, 11, 12],
       [ 8,  9, 10, 11, 12, 13, 14],
       [10, 11, 12, 13, 14, 15, 16],
       [12, 13, 14, 15, 16, 17, 18],
       [14, 15, 16, 17, 18, 19, 20]], dtype=int64)

In [30]:
boundary_condition_functions_source = [ga.wrap_or_invalidate_indices for ga in grid_level_one.axes]
neighbor_functions_individual_axes = []
for idx_cartesian in range(grid_level_one.ndim):
    nb_fun = partial(
        get_neigbhor_inds_on_fine_axis,
        axis_source_fine=grid_level_one.axes[idx_cartesian],
        axis_target_coarse=grid_level_two.axes[idx_cartesian],
    )
    neighbor_functions_individual_axes.append(nb_fun)
    
vmapped_neighbor_functions_individual_axes = [jax.vmap(nb_fun) for nb_fun in neighbor_functions_individual_axes]
    
indices_target_individual_axes = [jnp.arange(s) for s in grid_level_two.shape]

neighbor_inds_sourcegrid_individual_axes = [
    boundary_fun(vmapped_nb_fun(inds))
    for boundary_fun, vmapped_nb_fun, inds in zip(
        boundary_condition_functions_source,
        vmapped_neighbor_functions_individual_axes,
        indices_target_individual_axes,
    )
]

In [31]:
neighbor_inds_sourcegrid_individual_axes

[Array([[15, 15, 15, 15, 15, 15,  0],
        [15, 15, 15, 15,  0,  1,  2],
        [15, 15,  0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4,  5,  6],
        [ 2,  3,  4,  5,  6,  7,  8],
        [ 4,  5,  6,  7,  8,  9, 10],
        [ 6,  7,  8,  9, 10, 11, 12],
        [ 8,  9, 10, 11, 12, 13, 14],
        [10, 11, 12, 13, 14, 15, 15],
        [12, 13, 14, 15, 15, 15, 15],
        [14, 15, 15, 15, 15, 15, 15]], dtype=int64),
 Array([[15, 15, 15, 15, 15, 15,  0],
        [15, 15, 15, 15,  0,  1,  2],
        [15, 15,  0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4,  5,  6],
        [ 2,  3,  4,  5,  6,  7,  8],
        [ 4,  5,  6,  7,  8,  9, 10],
        [ 6,  7,  8,  9, 10, 11, 12],
        [ 8,  9, 10, 11, 12, 13, 14],
        [10, 11, 12, 13, 14, 15, 15],
        [12, 13, 14, 15, 15, 15, 15],
        [14, 15, 15, 15, 15, 15, 15]], dtype=int64),
 Array([[15, 15, 15, 15, 15, 15,  0],
        [15, 15, 15, 15,  0,  1,  2],
        [15, 15,  0,  1,  2,  3,  4],
        [ 0,  1,  2,

In [32]:
jax.vmap(make_multi_indices_one_particle)(*neighbor_inds_sourcegrid_individual_axes).shape

(11, 343, 3)

In [33]:
neighbor_inds_x_0 = neighbor_inds_sourcegrid_individual_axes[0][0]
neighbor_inds_y_0 = neighbor_inds_sourcegrid_individual_axes[1][0]
neighbor_inds_z_0 = neighbor_inds_sourcegrid_individual_axes[2][0]

neighbor_multi_indices_one_particle = make_multi_indices_one_particle(neighbor_inds_x_0, neighbor_inds_y_0, neighbor_inds_z_0)

In [34]:
neighbor_multi_indices_one_particle.shape

(343, 3)

In [35]:
# is_out_of_bounds = jnp.greater_equal(neighbor_multi_indices_one_particle, grid_level_one.shape[0])

In [36]:
# shape = jnp.array(grid_level_one.shape)
# is_not_periodic = ~jnp.array([ga.periodic for ga in grid_level_one.axes])
# # TODO: technically, what I need to check here is only < 0
# is_out_of_bounds = jnp.logical_or(neighbor_multi_indices_one_particle <0, neighbor_multi_indices_one_particle >= shape)
# is_out_of_bounds = is_out_of_bounds & is_not_periodic

In [37]:
[nb for nb in neighbor_functions_individual_axes]

[functools.partial(<function get_neigbhor_inds_on_fine_axis at 0x7fabf415a3a0>, axis_source_fine=BSplineInterpolationAxis(length=10.0, h=1.25, p=6, J_zeroplus=array([0.625  , 0.46875, 0.1875 , 0.03125]), periodic=False, n_domain=9, n_total=15, to_raw_indices=<function set_up_grid_axis.<locals>.to_raw_indices at 0x7fac0613fca0>, from_raw_indices=<function set_up_grid_axis.<locals>.from_raw_indices at 0x7fac0613fd30>, wrap_indices_if_periodic=<function set_up_grid_axis.<locals>.wrap_indices_if_periodic at 0x7fac590a40d0>, wrap_or_invalidate_indices=<function set_up_grid_axis.<locals>.wrap_or_invalidate_indices at 0x7fac590a4310>, evaluate_bspline_basis_for_one_particle=<function set_up_grid_axis.<locals>.evaluate_bspline_basis_for_one_particle at 0x7fac590a41f0>, evaluate_bspline_basis_multi=<function set_up_grid_axis.<locals>.evaluate_bspline_basis_multi at 0x7fac590a43a0>, evaluate_bspline_basis_gradient_multi=<function set_up_grid_axis.<locals>.evaluate_bspline_basis_gradient_multi at

In [38]:
def make_ravel_multi_inds_and_apply_bcs(grid: BSplineInterpolationGrid):
    shape = jnp.array(grid.shape)
    is_not_periodic = ~jnp.array([ga.periodic for ga in grid.axes])
    intentionally_out_of_bounds_index = grid.size
    
    def ravel_multi_inds_and_apply_bcs(multi_indices: jax.Array) -> jax.Array:
        # This handles periodic axes on its own due to the "wrap" keyword        
        flat_inds = jax.vmap(
            lambda multi_index: jnp.ravel_multi_index(
                multi_index, dims=shape, mode="wrap"
            )
        )(multi_indices)
        # Explicitly handle non-periodic axes
        is_out_of_bounds = jnp.logical_or(multi_indices < 0, multi_indices >= shape)
        is_out_of_bounds = (is_out_of_bounds & is_not_periodic).any(axis=1)
        flat_inds = jnp.where(is_out_of_bounds.ravel(), intentionally_out_of_bounds_index, flat_inds)

        return flat_inds

    return ravel_multi_inds_and_apply_bcs

In [39]:
ravel_multi_inds_and_apply_bcs = make_ravel_multi_inds_and_apply_bcs(grid_level_one)

In [40]:
neighbor_multi_indices_one_particle.shape

(343, 3)

In [41]:
grid_level_two.shape

(11, 11, 11)

In [42]:
neighbor_inds_sourcegrid_individual_axes

[Array([[15, 15, 15, 15, 15, 15,  0],
        [15, 15, 15, 15,  0,  1,  2],
        [15, 15,  0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4,  5,  6],
        [ 2,  3,  4,  5,  6,  7,  8],
        [ 4,  5,  6,  7,  8,  9, 10],
        [ 6,  7,  8,  9, 10, 11, 12],
        [ 8,  9, 10, 11, 12, 13, 14],
        [10, 11, 12, 13, 14, 15, 15],
        [12, 13, 14, 15, 15, 15, 15],
        [14, 15, 15, 15, 15, 15, 15]], dtype=int64),
 Array([[15, 15, 15, 15, 15, 15,  0],
        [15, 15, 15, 15,  0,  1,  2],
        [15, 15,  0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4,  5,  6],
        [ 2,  3,  4,  5,  6,  7,  8],
        [ 4,  5,  6,  7,  8,  9, 10],
        [ 6,  7,  8,  9, 10, 11, 12],
        [ 8,  9, 10, 11, 12, 13, 14],
        [10, 11, 12, 13, 14, 15, 15],
        [12, 13, 14, 15, 15, 15, 15],
        [14, 15, 15, 15, 15, 15, 15]], dtype=int64),
 Array([[15, 15, 15, 15, 15, 15,  0],
        [15, 15, 15, 15,  0,  1,  2],
        [15, 15,  0,  1,  2,  3,  4],
        [ 0,  1,  2,

In [43]:
i_x = 0
i_y = 1
i_z = 2

idx_gridpoint = 5

neighbor_multi_indices_one_particle = make_multi_indices_one_particle(
    neighbor_inds_sourcegrid_individual_axes[i_x][idx_gridpoint],
    neighbor_inds_sourcegrid_individual_axes[i_y][idx_gridpoint],
    neighbor_inds_sourcegrid_individual_axes[i_z][idx_gridpoint],
)

processed_flat_inds = ravel_multi_inds_and_apply_bcs(neighbor_multi_indices_one_particle)
processed_flat_inds

Array([ 964,  965,  966,  967,  968,  969,  970,  979,  980,  981,  982,
        983,  984,  985,  994,  995,  996,  997,  998,  999, 1000, 1009,
       1010, 1011, 1012, 1013, 1014, 1015, 1024, 1025, 1026, 1027, 1028,
       1029, 1030, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1054, 1055,
       1056, 1057, 1058, 1059, 1060, 1189, 1190, 1191, 1192, 1193, 1194,
       1195, 1204, 1205, 1206, 1207, 1208, 1209, 1210, 1219, 1220, 1221,
       1222, 1223, 1224, 1225, 1234, 1235, 1236, 1237, 1238, 1239, 1240,
       1249, 1250, 1251, 1252, 1253, 1254, 1255, 1264, 1265, 1266, 1267,
       1268, 1269, 1270, 1279, 1280, 1281, 1282, 1283, 1284, 1285, 1414,
       1415, 1416, 1417, 1418, 1419, 1420, 1429, 1430, 1431, 1432, 1433,
       1434, 1435, 1444, 1445, 1446, 1447, 1448, 1449, 1450, 1459, 1460,
       1461, 1462, 1463, 1464, 1465, 1474, 1475, 1476, 1477, 1478, 1479,
       1480, 1489, 1490, 1491, 1492, 1493, 1494, 1495, 1504, 1505, 1506,
       1507, 1508, 1509, 1510, 1639, 1640, 1641, 16

In [44]:
gridcharge_level_one.shape

(3375,)

In [45]:
gridcharge_level_one[processed_flat_inds]

Array([-0.06295449, -0.04081282, -0.03543972, -0.06020046, -0.01746742,
       -0.00527633, -0.01238504, -0.12345914, -0.0663086 , -0.0207792 ,
        0.00111847,  0.00306243, -0.01524047, -0.02886074, -0.03697446,
       -0.05621617, -0.0823024 , -0.01158802, -0.00732851, -0.05110701,
       -0.08493509, -0.00882213, -0.06420126, -0.13196647, -0.03416401,
        0.00458319, -0.00157451, -0.02958306,  0.01632992, -0.03060243,
       -0.09012046, -0.0268965 ,  0.01108062,  0.03525617,  0.00895554,
        0.02789845, -0.02162881, -0.07213203, -0.0264465 , -0.01338711,
       -0.01975534, -0.04233835,  0.00915135, -0.0038886 , -0.01797795,
       -0.02040641, -0.00336807, -0.00998259, -0.03313776, -0.02523725,
       -0.03333441, -0.01696205,  0.00367434,  0.00259954, -0.03548695,
       -0.08728673, -0.05426119, -0.03503847,  0.00579353,  0.07457742,
        0.044494  , -0.00908721, -0.06097514, -0.01379973, -0.01540526,
        0.00236374,  0.07514027,  0.05180029, -0.01204797, -0.09

In [46]:
multi_inds_target = make_multi_indices_one_particle(*[jnp.arange(s) for s in grid_level_two.shape])
flat_inds_target = make_ravel_multi_inds_and_apply_bcs(grid_level_two)(multi_inds_target)

In [47]:
multi_inds_target.shape, flat_inds_target.shape

((1331, 3), (1331,))

In [48]:
neighbor_inds_source_individual_axes = [nb_fun(idx) for nb_fun, idx in zip(neighbor_functions_individual_axes, multi_inds_target[100])]
make_multi_indices_one_particle(*neighbor_inds_source_individual_axes).shape

(343, 3)

In [49]:
def get_neighbor_multi_inds_one_gridpoint(multi_idx_target):
    neighbor_inds_source_individual_axes = [nb_fun(idx) for nb_fun, idx in zip(neighbor_functions_individual_axes, multi_idx_target)]
    return make_multi_indices_one_particle(*neighbor_inds_source_individual_axes)

In [50]:
def get_neighbor_flat_inds_one_gridpoint(multi_idx_target):
    neighbor_inds_source_individual_axes = [nb_fun(idx) for nb_fun, idx in zip(neighbor_functions_individual_axes, multi_idx_target)]
    multi_inds_source = make_multi_indices_one_particle(*neighbor_inds_source_individual_axes)
    return ravel_multi_inds_and_apply_bcs(multi_inds_source)

In [51]:
get_neighbor_multi_inds_one_gridpoint(multi_inds_target[0]).shape

(343, 3)

In [52]:
jax.vmap(get_neighbor_multi_inds_one_gridpoint)(multi_inds_target).shape

(1331, 343, 3)

In [53]:
get_neighbor_flat_inds_one_gridpoint(multi_inds_target[100])

Array([3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375, 3375,
       3375, 3375, 3375, 3375, 3375, 3375, 3375, 33

In [54]:
flat_neighbor_inds_all_target_gridpoints = jax.vmap(get_neighbor_flat_inds_one_gridpoint)(multi_inds_target)
J_multidim_flat = arbitrary_dim_outer(*([J] * grid_level_one.ndim)).ravel()

In [55]:
(flat_neighbor_inds_all_target_gridpoints * J_multidim_flat).shape

(1331, 343)

In [56]:
(flat_neighbor_inds_all_target_gridpoints * J_multidim_flat).sum(axis=1).shape

(1331,)

In [57]:
neighbor_multi_inds_all_target_points = jax.vmap(make_multi_indices_one_particle)(*neighbor_inds_sourcegrid_individual_axes)

In [58]:
neighbor_multi_inds_all_target_points.shape

(11, 343, 3)

In [59]:
(neighbor_multi_indices_one_particle == 0).sum() + (neighbor_multi_indices_one_particle == 15).sum()

Array(0, dtype=int64)

In [60]:
neighbor_multi_indices_one_particle.size

1029

In [61]:
neighbor_inds_sourcegrid = jax.vmap(make_multi_indices_one_particle)(*neighbor_inds_sourcegrid_individual_axes)
neighbor_inds_sourcegrid.shape

(11, 343, 3)

In [62]:
arr_on_source_grid = jnp.zeros(grid_level_one.shape)

In [63]:
arr_on_source_grid[neighbor_inds_sourcegrid[0]].shape

(343, 3, 15, 15)

In [64]:
neighbor_inds_sourcegrid[0]

Array([[15, 15, 15],
       [15, 15, 15],
       [15, 15, 15],
       ...,
       [ 0,  0, 15],
       [ 0,  0, 15],
       [ 0,  0,  0]], dtype=int64)

In [65]:
arr_on_source_grid[(0, 1, 2)]

Array(0., dtype=float64)

In [66]:
arr = jnp.outer(jnp.arange(1, 4), jnp.arange(1, 4))

In [67]:
arr

Array([[1, 2, 3],
       [2, 4, 6],
       [3, 6, 9]], dtype=int64)

In [68]:
@jax.jit
def myfun(x, inds):
    return 2 * x.at[inds].get(mode="fill", fill_value=0.0)

In [69]:
myfun(arr, inds=(0, 2))

Array(6, dtype=int64)

In [70]:
big_arr = onp.full((6, 6), -1)
for i in range(big_arr.shape[0]):
    for j in range(big_arr.shape[1]):
        big_arr[i, j] = myfun(arr, inds=(i, j))

In [71]:
jnp.asarray(big_arr)

Array([[ 2,  4,  6,  0,  0,  0],
       [ 4,  8, 12,  0,  0,  0],
       [ 6, 12, 18,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0]], dtype=int64)

In [72]:
restrict_1to2 = create_restriction_operator(grid_source_fine=grid_level_one, grid_target_coarse=grid_level_two)

In [73]:
gridcharge_level_two_from_restriction = restrict_1to2(gridcharge_level_one)

In [74]:
anterpolate_level_two = create_anterpolation_operator(grid_level_two)
gridcharge_level_two = anterpolate_level_two(pos, chg)

In [75]:
jnp.allclose(gridcharge_level_two, gridcharge_level_two_from_restriction)

Array(True, dtype=bool)

In [76]:
jitted_anterpolate_level_two = jax.jit(anterpolate_level_two)
jitted_restrict_1to2 = jax.jit(restrict_1to2)

In [79]:
jax.device_put(pos)
jax.device_put(chg)

%timeit jitted_anterpolate_level_two(pos, chg).block_until_ready()

318 µs ± 19.8 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [80]:
jax.device_put(gridcharge_level_one)

%timeit jitted_restrict_1to2(gridcharge_level_one).block_until_ready()

121 µs ± 894 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
